# Trần Minh Khoa - 20235121

In [36]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import time

# 1. LeNet-5

In [37]:
# LeNet5

class LeNet5Custom(nn.Module):
    def __init__(self,
                 in_channels=3,
                 num_classes=10,
                 filter_size_1=6,
                 filter_size_2=16,
    ):
        super(LeNet5Custom, self).__init__()

        ### 1. Feature extraction
        self.conv1 = nn.Conv2d(
            in_channels=in_channels,
            out_channels=filter_size_1,
            kernel_size=5,
            stride=1,
            padding=0,
        )
        self.pool = nn.MaxPool2d(
            kernel_size=2,
            stride=2,
        )
        self.conv2 = nn.Conv2d(
            in_channels=filter_size_1,
            out_channels=filter_size_2,
            kernel_size=5,
            stride=1,
            padding=0,
        )

        ### 2. Classifier
        # 32x32 -> Conv1: 28x28 -> Pool: 14x14 -> Conv2: 10x10 -> Pool: 5x5.
        self.fc1 = nn.Linear(
            in_features=filter_size_2 * 5 * 5, out_features=120
        )
        self.fc2 = nn.Linear(in_features=120, out_features=84)
        self.fc3 = nn.Linear(in_features=84, out_features=num_classes)

    def forward(self, x):

        x = self.pool(F.relu(self.conv1(x)))

        x = self.pool(F.relu(self.conv2(x)))

        x = x.view(x.size(0), -1)

        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))

        x = self.fc3(x)

        return x

# 2. VGG-16

In [39]:
# VGG16

class VGG16Custom(nn.Module):
    def __init__(self, in_channels=3, num_classes=10):
        super(VGG16Custom, self).__init__()

        ### 1. Feature Extraction
        # Conv 3x3, Padding 1, Stride 1 -> Same size in - out

        # Block 1: 32x32 -> Conv -> 32x32 -> Pool -> 16x16
        self.block1 = nn.Sequential(
            nn.Conv2d(in_channels=in_channels, out_channels=64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        # Block 2: 16x16 -> Conv -> 16x16 -> Pool -> 8x8 (Filter x2)
        self.block2 = nn.Sequential(
            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels=128, out_channels=128, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        # Block 3: 8x8 -> Conv -> 8x8 -> Pool -> 4x4 (Filter x2)
        self.block3 = nn.Sequential(
            nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels=256, out_channels=256, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels=256, out_channels=256, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        # Block 4: 4x4 -> Conv -> 4x4 -> Pool -> 2x2 (Filter x2)
        self.block4 = nn.Sequential(
            nn.Conv2d(in_channels=256, out_channels=512, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels=512, out_channels=512, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels=512, out_channels=512, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        # Block 5: 2x2 -> Conv -> 2x2 -> Pool -> 1x1 (Filter x2)
        self.block5 = nn.Sequential(
            nn.Conv2d(in_channels=512, out_channels=512, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels=512, out_channels=512, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels=512, out_channels=512, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        ### 2. Classifier
        # 32x32 -> 5 x MaxPool -> 1x1 --> 512 x 1 x 1

        self.classifier = nn.Sequential(
            nn.Linear(in_features=512 * 1 * 1, out_features=4096),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.5),

            nn.Linear(in_features=4096, out_features=4096),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.5),

            nn.Linear(in_features=4096, out_features=num_classes),
        )

    def forward(self, x):

        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.block5(x)

        # -1 -> 512 x 1 x 1
        x = x.view(x.size(0), -1)

        x = self.classifier(x)

        return x

# 3. GoogleNet

In [41]:
# GoogleNet

class InceptionModule(nn.Module):

    def __init__(self, in_channels, n1x1, n3x3_reduce, n3x3, n5x5_reduce, n5x5, pool_proj):
        super(InceptionModule, self).__init__()

        ### Branch 1: Conv 1x1
        self.branch1 = nn.Sequential(
            nn.Conv2d(in_channels=in_channels, out_channels=n1x1, kernel_size=1),
            nn.ReLU(inplace=True),
        )

        ### Branch 2: Conv 1x1_reduce -> 3x3
        self.branch2 = nn.Sequential(
            nn.Conv2d(in_channels=in_channels, out_channels=n3x3_reduce, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels=n3x3_reduce, out_channels=n3x3, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
        )

        ### Branch 3: Conv 1x1_reduce -> 5x5
        self.branch3 = nn.Sequential(
            nn.Conv2d(in_channels=in_channels, out_channels=n5x5_reduce, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels=n5x5_reduce, out_channels=n5x5, kernel_size=5, padding=2),
            nn.ReLU(inplace=True),
        )

        ### Branch 4: MaxPool 3x3 -> Conv 1x1
        self.branch4 = nn.Sequential(
            nn.MaxPool2d(kernel_size=3, stride=1, padding=1),
            nn.Conv2d(in_channels=in_channels, out_channels=pool_proj, kernel_size=1),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):

        out1 = self.branch1(x)
        out2 = self.branch2(x)
        out3 = self.branch3(x)
        out4 = self.branch4(x)

        return torch.cat([out1, out2, out3, out4], dim=1)

class GoogleNetCustom(nn.Module):
    def __init__(self, num_classes=10):
        super(GoogleNetCustom, self).__init__()

        ### 1. Stem Layer
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=192, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )

        ### 2. Inception Modules
        self.inception_3a = InceptionModule(in_channels=192, n1x1=64, n3x3_reduce=96, n3x3=128, n5x5_reduce=16, n5x5=32, pool_proj=32)
        self.inception_3b = InceptionModule(in_channels=256, n1x1=128, n3x3_reduce=128, n3x3=192, n5x5_reduce=32, n5x5=96, pool_proj=64)

        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        ### 3. Global Average Pooling
        self.avgpool = nn.AdaptiveAvgPool2d(output_size=(1, 1))
        self.dropout = nn.Dropout(p=0.4)
        self.fc = nn.Linear(in_features=480, out_features=num_classes) # 480: after 3b

    def forward(self, x):
        x = self.stem(x)
        x = self.inception_3a(x)
        x = self.inception_3b(x)
        x = self.maxpool(x)

        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        x = self.fc(x)
        return x

# 4. ResNet-50

In [42]:
# ResNet-50

class Bottleneck(nn.Module):

    # Expansion factor in final layer of module (Always x4)
    expansion = 4

    def __init__(self, in_channels, out_channels, stride=1, downsample=None):
        super(Bottleneck, self).__init__()

        ### 1. Conv 1x1: downsize
        self.conv1 = nn.Conv2d(in_channels=in_channels, out_channels=out_channels, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(num_features=out_channels)

        ### 2. Conv 3x3: learned feature
        self.conv2 = nn.Conv2d(in_channels=out_channels, out_channels=out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(num_features=out_channels)

        ### 3. Conv 1x1: resize
        self.conv3 = nn.Conv2d(out_channels, out_channels * self.expansion, kernel_size=1, bias=False)
        self.bn3 = nn.BatchNorm2d(out_channels * self.expansion)

        self.relu = nn.ReLU(inplace=True)
        self.downsample = downsample

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)
        out = self.relu(out)

        out = self.conv3(out)
        out = self.bn3(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        out = self.relu(out)

        return out


class ResNet50Custom(nn.Module):
    def __init__(self, num_classes=10):
        super(ResNet50Custom, self).__init__()

        self.in_channels = 64

        ### Stem Layer
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(num_features=64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        ### 4 stages
        self.layer1 = self._make_layer(64, 3)
        self.layer2 = self._make_layer(128, 4, stride=2)
        self.layer3 = self._make_layer(256, 6, stride=2)
        self.layer4 = self._make_layer(512, 3, stride=2)

        ### Global Average Pooling
        self.avgpool = nn.AdaptiveAvgPool2d(output_size=(1, 1))
        self.fc = nn.Linear(in_features=512 * Bottleneck.expansion, out_features=num_classes)

    def _make_layer(self, out_channels, blocks, stride=1):
        downsample = None

        # If stride != 1 or in_channels != out_channels, create a downsample layer
        if stride != 1 or self.in_channels != out_channels * Bottleneck.expansion:
            downsample = nn.Sequential(
                nn.Conv2d(in_channels=self.in_channels, out_channels=out_channels * Bottleneck.expansion, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(num_features=out_channels * Bottleneck.expansion),
            )

        layers = []
        layers.append(Bottleneck(self.in_channels, out_channels, stride, downsample))
        self.in_channels = out_channels * Bottleneck.expansion
        for _ in range(1, blocks):
            layers.append(Bottleneck(self.in_channels, out_channels))

        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.maxpool(self.relu(self.bn1(self.conv1(x))))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x

# Training

In [45]:
# --- 1. SET DEVICE ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# --- 2. DATA PREPARATION (CIFAR-10) ---
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

train_set = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test_set = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_set, batch_size=128, shuffle=True, num_workers=2)
test_loader = DataLoader(test_set, batch_size=128, shuffle=False, num_workers=2)

# --- 3. TRAINING & EVALUATION FUNCTION ---
def train_model(model, model_name, epochs=5):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    history = {'loss': [], 'acc': []}
    print(f"\n>>> Starting training: {model_name}")

    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        correct = 0
        total = 0
        start_time = time.time()

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            # Forward pass
            outputs = model(images)
            loss = criterion(outputs, labels)

            # Backward and optimize
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # Statistics
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

        epoch_loss = running_loss / len(train_loader)
        epoch_acc = 100. * correct / total
        history['loss'].append(epoch_loss)
        history['acc'].append(epoch_acc)

        duration = time.time() - start_time
        print(f"Epoch [{epoch+1}/{epochs}] | Loss: {epoch_loss:.4f} | Acc: {epoch_acc:.2f}% | Time: {duration:.2f}s")

    return history

# --- 4. LIST OF MODELS TO RUN ---
models_to_train = [
    (LeNet5Custom(), "LeNet-5"),
    (VGG16Custom(), "VGG-16"),
    (GoogleNetCustom(), "GoogLeNet"),
    (ResNet50Custom(), "ResNet-50")
]

all_histories = {}

# --- 5. EXECUTION LOOP ---
for model_obj, name in models_to_train:
    all_histories[name] = train_model(model_obj, name, epochs=10)

# --- 6. PLOTTING RESULTS ---
plt.figure(figsize=(12, 5))

# Plot Accuracy
plt.subplot(1, 2, 1)
for name, hist in all_histories.items():
    plt.plot(hist['acc'], label=name)
plt.title('Training Accuracy Comparison')
plt.xlabel('Epochs')
plt.ylabel('Accuracy (%)')
plt.legend()

# Plot Loss
plt.subplot(1, 2, 2)
for name, hist in all_histories.items():
    plt.plot(hist['loss'], label=name)
plt.title('Training Loss Comparison')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

Using device: cpu

>>> Starting training: LeNet-5
Epoch [1/10] | Loss: 1.6706 | Acc: 39.03% | Time: 3.87s
Epoch [2/10] | Loss: 1.3615 | Acc: 50.98% | Time: 4.39s
Epoch [3/10] | Loss: 1.2320 | Acc: 56.10% | Time: 4.95s
Epoch [4/10] | Loss: 1.1451 | Acc: 59.61% | Time: 5.48s
Epoch [5/10] | Loss: 1.0845 | Acc: 61.86% | Time: 3.61s
Epoch [6/10] | Loss: 1.0234 | Acc: 64.21% | Time: 5.01s
Epoch [7/10] | Loss: 0.9763 | Acc: 65.61% | Time: 4.52s
Epoch [8/10] | Loss: 0.9345 | Acc: 67.10% | Time: 4.63s
Epoch [9/10] | Loss: 0.8898 | Acc: 68.68% | Time: 5.02s
Epoch [10/10] | Loss: 0.8593 | Acc: 69.80% | Time: 4.84s

>>> Starting training: VGG-16


KeyboardInterrupt: 